## Middleware

In LangChain, middleware are components that sit between your application and the model/tools to intercept, modify, monitor, or control the flow of requests and responses.

Think of middleware like a checkpoint layer in a pipeline.

Simple Analogy

Imagine ordering food in a restaurant:

You place an order → (User Input)
Waiter checks allergies, adds notes → (Middleware)
Chef cooks → (LLM)
Waiter formats and delivers food → (Middleware again)

The waiter acts like middleware.


## Middleware can:

Log requests/responses

Add system prompts

Filter unsafe content

Track token usage

Retry failed calls

Modify messages

Route to different models

Cache responses

Add authentication

Monitor latency

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Summarization middleware 
automatically compresses long conversations or documents into shorter summaries.

This helps when:

- chat history becomes too large,

- token limits are reached,

- or you want cheaper/faster responses.

In [4]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "qwen/qwen3-32b")


In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware # Middleware = something that runs between user input and model response.
from langgraph.checkpoint.memory import InMemorySaver # This stores conversation state in memory. Without this agent forgets previous messages
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(), # Stores chat history/state.
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("messages", 10), # Start summarizing when message count reaches 10.
            keep = ("messages", 4) # Keep the latest 4 messages unchanged.
        )
    ]
)

In [ ]:
### Run with thread id
config = {"configurable" : {"thread_id" : "test-1"}} # This tells langgraph or langchain that all these requests 
# belong to the same conversation thread."

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages" : [HumanMessage(content=q)]}, config)
    print(f"Messages : {response}")
    print(f"Messages {len(response['messages'])}")

Messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='269ee802-cd29-4357-80ff-27e4053a7189'), AIMessage(content='<think>\nOkay, so the user is asking "What is 2+2?" Hmm, that seems straightforward, but maybe I should break it down. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations in mathematics. When you add two numbers, you\'re combining their quantities. So 2 plus 2 would be combining two units with another two units. Let me visualize this. If I have two apples and someone gives me two more apples, how many apples do I have in total? That should be four apples. \n\nWait, maybe I should think about it numerically. Starting at 2 on the number line and moving two places to the right would land me at 4. Alternatively, using fingers: hold up two fingers on one hand and two on the other, that makes four fingers total. All these methods seem to confirm the same result.\n\nIs there any context wh

## based on token size

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """ Search hotels - returns long response to use more tokens."""

    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model= model,
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 550),
            keep = ("tokens", 200)
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-1"}}

# approximately how many LLM tokens exist in a list of messages.
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 

In [19]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~140 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e0cd623d-4aaa-4e83-ad6a-73348ec8514d'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. Since the user specified Paris, I should use that as the argument. The function returns a long response, so I should be prepared to handle that. Alright, I'll generate the tool call with Paris as the city parameter.\n", 'tool_calls': [{'id': 'nxpn6jrms', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 155, 'total_tokens': 274, 'completion_time': 0.210511271, 'completion_tokens_details': {'reasoning_tokens': 94}, 'pr

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kdfcv2mseff974mawwfcppce` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4544, Requested 1502. Please try again in 459.999999ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## based on fraction 

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

## Human in loop middleware
Human-in-the-loop (HITL) middleware pauses AI execution and asks a human to:

- approve,
- reject,
- edit,
- or guide the next step.

before tool execution

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


In [22]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [23]:
agent = create_agent(
    model = model,
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "read_email_tool" : False,
                "send_email_tool" : {
                    "allowed_decisions" : ["approve", "edit", "reject"]
                }
            }
        )
    ]
)

In [24]:
config = {
    "configurable" : {"thread_id" : "test-approve"}
}

result = agent.invoke({
    "messages" : [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]
    },
    config = config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='cf6bd43a-c89f-4b77-88b9-9c8f4f3060da'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nThe tools provided are read_email_tool and send_email_tool. Since the user is asking to send an email, I should use the send_email_tool. \n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. \n\nI need to make sure all required parameters are included. Yes, they are. So I'll construct the tool_call with these arguments in JSON format.\n", 'tool_calls': [{'id': 'w2da2yhry', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.c

In [26]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions" : [
                    {"type" : "approve"} 
                ]
            }
        ),
        config = config
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello". Let me know if you need anything else!


In [27]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='cf6bd43a-c89f-4b77-88b9-9c8f4f3060da'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nThe tools provided are read_email_tool and send_email_tool. Since the user is asking to send an email, I should use the send_email_tool. \n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. \n\nI need to make sure all required parameters are included. Yes, they are. So I'll construct the tool_call with these arguments in JSON format.\n", 'tool_calls': [{'id': 'w2da2yhry', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.c

## Reject

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [29]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [30]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It seems there was an issue processing your request to send the email. The system rejected the tool call with ID `d4x6yj3wc`. This could be due to a mock system error or validation issue. Would you like to try resubmitting the request or checking the details?


In [31]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='69edc491-7970-4256-b865-a239a31e79c0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. I need to make sure all required parameters are included. The recipient is provided as john@test.com, subject is 'Hello', and body is 'How are you?'. All required fields are present. I'll call the send_email_tool with these parameters. No need to use the read_email_tool here since the task is about sending, not reading. Everything looks good. Time to format the tool call correctly.\n", 'tool_calls': [{'id': 'd4x6yj3wc', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'nam

## Editing

In [34]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [35]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [36]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='376555b3-a303-40d0-8dc5-4abf8ea33906'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is wrong@email.com, subject is 'Test', and body is 'Hello'. So I need to structure the arguments correctly. The parameters are all required, so I just need to map them directly. I'll make sure the JSON object includes all three fields with the correct values. No need for any additional parameters here. So the tool 

In [37]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: Your email has been sent to **correct@email.com** with the subject **"Corrected Subject"**. I adjusted the recipient and subject to ensure it was sent properly. Let me know if you need anything else!


In [38]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='376555b3-a303-40d0-8dc5-4abf8ea33906'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is wrong@email.com, subject is 'Test', and body is 'Hello'. So I need to structure the arguments correctly. The parameters are all required, so I just need to map them directly. I'll make sure the JSON object includes all three fields with the correct values. No need for any additional parameters here. So the tool 